List every CSV file

In [1]:
from pathlib import Path

# Find the project directory
PROJECT_ROOT = Path.cwd().parent

# Raw data directory
RAW_DIR = PROJECT_ROOT / "DATA" / "RAW"

# Display all CSV files and their sizes
for file in sorted(RAW_DIR.rglob("*.csv")):
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.relative_to(RAW_DIR)} | {size_mb:.2f} MB")

companies\companies.csv | 22.12 MB
companies\company_industries.csv | 0.75 MB
companies\company_specialities.csv | 4.23 MB
companies\employee_counts.csv | 1.00 MB
jobs\benefits.csv | 1.84 MB
jobs\job_industries.csv | 2.39 MB
jobs\job_skills.csv | 3.34 MB
jobs\salaries.csv | 2.15 MB
mappings\industries.csv | 0.01 MB
mappings\skills.csv | 0.00 MB
postings.csv | 492.90 MB


### Inspect the main dataset

In [3]:
import pandas as pd

postings = pd.read_csv(
    RAW_DIR / "postings.csv",
    nrows=5
)

print("Columns:")
print(postings.columns.tolist())

print("\nSample data:")
display(postings.head())

Columns:
['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']

Sample data:


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540,34021
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521,8069
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202,39061
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040,36059
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601,19057


## Calculate missing values

In [4]:
import pandas as pd

file_path = RAW_DIR / "postings.csv"

missing_counts = None
total_rows = 0

for chunk in pd.read_csv(file_path, chunksize=100_000):

    if missing_counts is None:
        missing_counts = pd.Series(
            0, index=chunk.columns, dtype="int64"
        )

    missing_counts += chunk.isna().sum()
    total_rows += len(chunk)

print("Total rows:", total_rows)
print("Total columns:", len(missing_counts))

Total rows: 123849
Total columns: 31


In [5]:
missing_report = pd.DataFrame({
    "Missing_Count": missing_counts,
    "Missing_Percentage": (
        missing_counts / total_rows * 100
    ).round(2)
})

missing_report = missing_report.sort_values(
    "Missing_Percentage",
    ascending=False
)

display(missing_report)

,Missing_Count,Missing_Percentage
closed_time,122776,99.13
skills_desc,121410,98.03
med_salary,117569,94.93
remote_allowed,108603,87.69
applies,100529,81.17
min_salary,94056,75.94
max_salary,94056,75.94
currency,87776,70.87
compensation_type,87776,70.87
pay_period,87776,70.87


### Find out how much usable salary data we have

In [6]:
salary_columns = [
    "min_salary",
    "max_salary",
    "med_salary",
    "normalized_salary"
]

salary_rows = 0
total_rows = 0
salary_column_counts = pd.Series(dtype="int64")

for chunk in pd.read_csv(
    RAW_DIR / "postings.csv",
    usecols=salary_columns,
    chunksize=100_000
):
    total_rows += len(chunk)

    salary_rows += chunk.notna().any(axis=1).sum()

    salary_column_counts = (
        salary_column_counts
        .add(chunk.notna().sum(), fill_value=0)
    )

print("Total postings:", total_rows)
print("Postings with any salary:", salary_rows)
print(
    "Salary availability (%):",
    round(salary_rows / total_rows * 100, 2)
)

print("\nNon-missing values by salary column:")
print(salary_column_counts.astype(int))

Total postings: 123849
Postings with any salary: 36073
Salary availability (%): 29.13

Non-missing values by salary column:
max_salary           29793
med_salary            6280
min_salary           29793
normalized_salary    36073
dtype: int64


### Check duplicate job postings

In [7]:
import pandas as pd

job_ids = pd.read_csv(
    RAW_DIR / "postings.csv",
    usecols=["job_id"]
)

print("Total rows:", len(job_ids))
print("Duplicate job IDs:", job_ids["job_id"].duplicated().sum())
print("Unique job IDs:", job_ids["job_id"].nunique())

Total rows: 123849
Duplicate job IDs: 0
Unique job IDs: 123849


In [8]:
seen_ids = set()
duplicate_ids = 0

for chunk in pd.read_csv(
    RAW_DIR / "postings.csv",
    usecols=["job_id"],
    chunksize=100_000
):
    duplicate_ids += chunk["job_id"].isin(seen_ids).sum()
    seen_ids.update(chunk["job_id"].dropna())

print("Repeated job IDs:", duplicate_ids)

Repeated job IDs: 0


### Inspect data types

In [9]:
import pandas as pd

columns_to_inspect = [
    "job_id",
    "title",
    "description",
    "min_salary",
    "max_salary",
    "med_salary",
    "normalized_salary",
    "pay_period",
    "currency",
    "listed_time",
    "remote_allowed",
    "formatted_experience_level"
]

df = pd.read_csv(
    RAW_DIR / "postings.csv",
    usecols=columns_to_inspect,
    nrows=1000
)

print(df.dtypes)

job_id                          int64
title                             str
description                       str
max_salary                    float64
pay_period                        str
med_salary                    float64
min_salary                    float64
remote_allowed                float64
formatted_experience_level        str
listed_time                   float64
currency                          str
normalized_salary             float64
dtype: object


### Check salary consistency

In [10]:
salary_check = df[
    ["min_salary", "max_salary", "med_salary", "normalized_salary"]
]

print("Salary columns:")
display(salary_check.describe())

print(
    "Rows where minimum salary exceeds maximum salary:",
    (df["min_salary"] > df["max_salary"]).sum()
)

print(
    "Rows with negative salary values:",
    (salary_check < 0).any(axis=1).sum()
)

Salary columns:


,min_salary,max_salary,med_salary,normalized_salary
count,338.000000,338.000000,42.000000,380.000000
mean,51232.463195,74186.792367,31165.679524,86904.391645
std,52429.404987,82059.032139,48364.396641,63552.899232
min,1.000000,15.000000,15.000000,47.500000
25%,35.222500,50.000000,22.250000,52000.000000
50%,50000.000000,72000.000000,50.865000,72500.000000
75%,76500.000000,105000.000000,60000.000000,99685.000000
max,420000.000000,700000.000000,200000.000000,624000.000000


Rows where minimum salary exceeds maximum salary: 0
Rows with negative salary values: 0


## Investigate salary units

In [11]:
# Check salary units and currencies
print("Pay period distribution:")
display(df["pay_period"].value_counts(dropna=False))

print("\nCurrency distribution:")
display(df["currency"].value_counts(dropna=False))

print("\nSalary records with low minimum values:")
display(
    df.loc[
        df["min_salary"].notna()
        & (df["min_salary"] < 1000),
        [
            "title",
            "min_salary",
            "max_salary",
            "med_salary",
            "pay_period",
            "currency",
            "normalized_salary"
        ]
    ].head(15)
)

Pay period distribution:


pay_period
NaN        620
YEARLY     237
HOURLY     130
MONTHLY     13
Name: count, dtype: int64


Currency distribution:


currency
NaN    620
USD    380
Name: count, dtype: int64


Salary records with low minimum values:


,title,min_salary,max_salary,med_salary,pay_period,currency,normalized_salary
0,Marketing Coordinator,17.0,20.0,NaN,HOURLY,USD,38480.0
1,Mental Health Therapist/Counselor,30.0,50.0,NaN,HOURLY,USD,83200.0
5,Economic Development and Planning Intern,14.0,20.0,NaN,HOURLY,USD,35360.0
17,Customer Service / Reservationist,18.0,19.0,NaN,HOURLY,USD,38480.0
28,General Laborer,18.0,24.0,NaN,HOURLY,USD,43680.0
45,TREC licensed Professional Home Inspector,24.0,28.0,NaN,HOURLY,USD,54080.0
62,Social Security Specialist / Retirement Benefi...,19.0,25.0,NaN,HOURLY,USD,45760.0
73,Mover/Junk Hauler/Crew Member,20.0,25.0,NaN,HOURLY,USD,46800.0
82,Sales and Design consultant,23.0,26.0,NaN,HOURLY,USD,50960.0
85,"Validation Engineer, Labware LIMS",60.0,70.0,NaN,HOURLY,USD,135200.0


What we've learned
- 620 of 1,000 rows have no salary information in this sample.
- All 380 rows with salary data use USD.
- The salary periods include YEARLY, HOURLY, and MONTHLY.
- An hourly salary of $17–$20 becomes $35,360–$41,600 annually in the normalized salary field, assuming 2,080 working hours per year.

## Investigate suspicious salary records
Let's identify records where the normalized salary doesn't appear consistent with the salary range and pay period.

In [12]:
# Show yearly salary records with unusually low minimum salaries
suspicious = df[
    (df["pay_period"] == "YEARLY")
    & (df["min_salary"].notna())
    & (df["min_salary"] < 1000)
]

display(
    suspicious[
        [
            "title",
            "min_salary",
            "max_salary",
            "pay_period",
            "currency",
            "normalized_salary"
        ]
    ]
)

,title,min_salary,max_salary,pay_period,currency,normalized_salary
125,"Director, Investment Sales",1.0,40000.0,YEARLY,USD,20000.5
191,Sales Development Representative,1.0,130813.0,YEARLY,USD,65407.0
266,Principal Backend Engineer,200.0,225.0,YEARLY,USD,212.5
308,Records Specialist & Academic Scheduler,45.0,50.0,YEARLY,USD,47.5
317,Product Support Specialist,70.0,100000.0,YEARLY,USD,50035.0
374,Financial Services Representative,5.0,250000.0,YEARLY,USD,125002.5
542,Commercial Real Estate Attorney,120.0,180.0,YEARLY,USD,150.0
590,Construction Superintendent,120.0,150.0,YEARLY,USD,135.0


In [13]:
salary_df = pd.read_csv(
    RAW_DIR / "postings.csv",
    usecols=[
        "job_id",
        "title",
        "min_salary",
        "max_salary",
        "med_salary",
        "normalized_salary",
        "pay_period",
        "currency"
    ]
)

print("Rows with minimum > maximum:")
print(
    (
        salary_df["min_salary"]
        > salary_df["max_salary"]
    ).sum()
)

print("\nSalary periods:")
print(salary_df["pay_period"].value_counts(dropna=False))

print("\nCurrency distribution:")
print(salary_df["currency"].value_counts(dropna=False))

Rows with minimum > maximum:
0

Salary periods:
pay_period
NaN         87776
YEARLY      20628
HOURLY      14741
MONTHLY       518
WEEKLY        177
BIWEEKLY        9
Name: count, dtype: int64

Currency distribution:
currency
NaN    87776
USD    36058
EUR        6
CAD        3
BBD        2
AUD        2
GBP        2
Name: count, dtype: int64


Let's inspect the suspicious salaries

In [14]:
# Focus on suspiciously low yearly salary ranges
yearly = salary_df[
    (salary_df["pay_period"] == "YEARLY")
    & salary_df["min_salary"].notna()
    & salary_df["max_salary"].notna()
].copy()

yearly["salary_range"] = (
    yearly["max_salary"] - yearly["min_salary"]
)

display(
    yearly.nsmallest(15, "normalized_salary")[
        [
            "title",
            "min_salary",
            "max_salary",
            "pay_period",
            "currency",
            "normalized_salary"
        ]
    ]
)

,title,min_salary,max_salary,pay_period,currency,normalized_salary
26910,ACCOUNT DIRECTOR SENIOR-ENTERPRISE 1,1.00,1.00,YEARLY,USD,1.000
10978,Regional Sales Manager,1.00,2.00,YEARLY,USD,1.500
70962,Resident Manager,12.00,14.00,YEARLY,USD,13.000
32439,Assistant Store Manager,14.50,15.25,YEARLY,USD,14.875
49614,Visitor Services Assistant Intern,15.00,15.00,YEARLY,USD,15.000
5111,Recreation Aid NF-01,15.00,16.79,YEARLY,USD,15.895
44188,Recreation Assistant (Facilities Operator) NF-02,15.00,17.00,YEARLY,USD,16.000
5024,"Recreation Aid (Lifeguard), NF-01",15.00,17.68,YEARLY,USD,16.340
5150,"Recreation Aid (Trainee-Lifeguard), NF-01",15.00,17.68,YEARLY,USD,16.340
78955,Recreation Aid NF-01,15.00,17.68,YEARLY,USD,16.340


Count suspicious yearly salaries

In [15]:
# Identify unusually low yearly salary records
low_yearly = yearly[
    (yearly["min_salary"] < 1000)
    | (yearly["max_salary"] < 1000)
]

print("Yearly records examined:", len(yearly))
print("Suspicious low-salary records:", len(low_yearly))

print("\nPay-period distribution of suspicious records:")
display(low_yearly["pay_period"].value_counts())

print("\nSalary summary of suspicious records:")
display(
    low_yearly[
        ["min_salary", "max_salary", "normalized_salary"]
    ].describe()
)

Yearly records examined: 19107
Suspicious low-salary records: 385

Pay-period distribution of suspicious records:


pay_period
YEARLY    385
Name: count, dtype: int64


Salary summary of suspicious records:


,min_salary,max_salary,normalized_salary
count,385.00000,385.000000,385.000000
mean,52.17974,7435.783688,3743.981714
std,38.55984,31870.301376,15935.266299
min,1.00000,1.000000,1.000000
25%,22.00000,28.000000,26.000000
50%,40.00000,50.000000,48.250000
75%,70.00000,95.000000,92.500000
max,200.00000,250000.000000,125002.500000


Create a salary-quality flag

In [16]:
# Flag suspicious yearly salaries
salary_df["salary_quality_flag"] = (
    (salary_df["pay_period"] == "YEARLY")
    & (
        (salary_df["min_salary"] < 1000)
        | (salary_df["max_salary"] < 1000)
    )
)

print(
    "Flagged records:",
    salary_df["salary_quality_flag"].sum()
)

print("\nFlagged examples:")
display(
    salary_df.loc[
        salary_df["salary_quality_flag"],
        [
            "title",
            "min_salary",
            "max_salary",
            "pay_period",
            "currency",
            "normalized_salary"
        ]
    ].head(10)
)

Flagged records: 385

Flagged examples:


,title,min_salary,max_salary,pay_period,currency,normalized_salary
125,"Director, Investment Sales",1.0,40000.0,YEARLY,USD,20000.5
191,Sales Development Representative,1.0,130813.0,YEARLY,USD,65407.0
266,Principal Backend Engineer,200.0,225.0,YEARLY,USD,212.5
308,Records Specialist & Academic Scheduler,45.0,50.0,YEARLY,USD,47.5
317,Product Support Specialist,70.0,100000.0,YEARLY,USD,50035.0
374,Financial Services Representative,5.0,250000.0,YEARLY,USD,125002.5
542,Commercial Real Estate Attorney,120.0,180.0,YEARLY,USD,150.0
590,Construction Superintendent,120.0,150.0,YEARLY,USD,135.0
1284,Data Entry Specialist,20.0,24.0,YEARLY,USD,22.0
1450,Embedded Engineer,110.0,140.0,YEARLY,USD,125.0


In [17]:
from pathlib import Path
import pandas as pd

# Read the original dataset
df = pd.read_csv(RAW_DIR / "postings.csv")

# Flag suspicious yearly salary records
df["salary_quality_flag"] = (
    (df["pay_period"] == "YEARLY")
    & (
        (df["min_salary"] < 1000)
        | (df["max_salary"] < 1000)
    )
)

# Create processed directory if it doesn't exist
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save without modifying the raw file
output_path = PROCESSED_DIR / "job_postings_flagged.csv"
df.to_csv(output_path, index=False)

print("Rows saved:", len(df))
print("Columns saved:", len(df.columns))
print("Flagged salary records:", df["salary_quality_flag"].sum())
print("Saved to:", output_path)

Rows saved: 123849
Columns saved: 32
Flagged salary records: 385
Saved to: d:\project\DS\Job Market & Salary Analysis project\data\processed\job_postings_flagged.csv
